In [ ]:
#Notebook description

# this notebook is intended to track and price commodities based on the demand and supply data

In [ ]:
#Load libraries 
import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
import sys
sys.path.append(r"e:\Coding Projects\Investment Analysis")

# Load libraries
from Quantapp.visualization import Plotter
from Quantapp.analytics import TimeSeriesAnalytics as Rolling
from Quantapp.data import MacroDataClient

import numpy as np
import json
import os
import pandas as pd
import yfinance as yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.graph_objects as go
import plotly.graph_objects as go
import pandas as pd
from plotly.subplots import make_subplots

#shut down warnings
import warnings
warnings.filterwarnings("ignore")


qc = Rolling()
qp = Plotter()
qe = MacroDataClient()

In [ ]:
#parameters
period = "20y"

In [ ]:
# TurtleTrader futures term structure
import io
import re
import urllib.request
import zipfile

TURTLE_MARKETS = {
    "Crude Oil": {"slug": "cl", "prefix": "CL"},
    "Heating Oil": {"slug": "ho", "prefix": "HO"},
    "Natural Gas": {"slug": "ng", "prefix": "NG"},
    "Gasoline": {"slug": "rb", "prefix": "RB"},
    "Gold": {"slug": "gc", "prefix": "GC"},
    "Silver": {"slug": "si", "prefix": "SI"},
    "Copper": {"slug": "hg", "prefix": "HG"},
    "Corn": {"slug": "c", "prefix": "C"},
    "Wheat": {"slug": "w", "prefix": "W"},
    "Soybeans": {"slug": "s", "prefix": "S"},
}

selected_turtle_market = "Crude Oil"  # Change this to any key in TURTLE_MARKETS.
max_contracts = 12
min_curve_points = 6
max_quote_age_days = 45

month_code_to_number = {
    "F": 1,
    "G": 2,
    "H": 3,
    "J": 4,
    "K": 5,
    "M": 6,
    "N": 7,
    "Q": 8,
    "U": 9,
    "V": 10,
    "X": 11,
    "Z": 12,
}

market_config = TURTLE_MARKETS[selected_turtle_market]
zip_url = f"https://www.turtletrader.com/cddata/{market_config['slug']}.zip"
request = urllib.request.Request(zip_url, headers={"User-Agent": "Mozilla/5.0"})
zip_bytes = urllib.request.urlopen(request, timeout=30).read()
archive = zipfile.ZipFile(io.BytesIO(zip_bytes))

def parse_contract_year(two_digit_year: int) -> int:
    current_two_digit_year = pd.Timestamp.now(tz="UTC").year % 100
    return 2000 + two_digit_year if two_digit_year <= current_two_digit_year + 5 else 1900 + two_digit_year

contract_pattern = re.compile(
    rf"^{market_config['prefix']}(?P<year>\d{{2}})(?P<month>[FGHJKMNQUVXZ])\.txt$",
    re.IGNORECASE,
)

contract_frames = []
for file_name in archive.namelist():
    match = contract_pattern.match(file_name.split("/")[-1])
    if not match:
        continue

    rows = [
        line.split(",")
        for line in archive.read(file_name).decode("utf-8", errors="ignore").splitlines()
        if line.strip()
    ]
    if not rows:
        continue

    contract_data = pd.DataFrame(
        rows,
        columns=["date", "open", "high", "low", "close", "volume", "open_interest"],
    )
    contract_data["date"] = pd.to_datetime(contract_data["date"], format="%y%m%d", errors="coerce")
    for column in ["open", "high", "low", "close", "volume", "open_interest"]:
        contract_data[column] = pd.to_numeric(contract_data[column], errors="coerce")
    contract_data = contract_data.dropna(subset=["date", "close"]).sort_values("date")
    if contract_data.empty:
        continue

    contract_year = parse_contract_year(int(match.group("year")))
    contract_month = month_code_to_number[match.group("month").upper()]
    contract_month_date = pd.Timestamp(year=contract_year, month=contract_month, day=1)

    contract_frames.append(
        contract_data.assign(
            contractCode=file_name.replace(".txt", ""),
            contractMonth=contract_month_date,
            contractLabel=contract_month_date.strftime("%b %Y"),
        )
    )

if not contract_frames:
    raise RuntimeError(f"No contract files were parsed from {zip_url}")

turtle_contract_history = pd.concat(contract_frames, ignore_index=True)
candidate_dates = sorted(turtle_contract_history["date"].dropna().unique(), reverse=True)

def build_term_structure(as_of_date):
    front_month_floor = pd.Timestamp(year=as_of_date.year, month=as_of_date.month, day=1)
    curve_rows = []

    for contract_code, contract_history in turtle_contract_history.groupby("contractCode"):
        quote = contract_history.loc[contract_history["date"] <= as_of_date].sort_values("date").tail(1)
        if quote.empty:
            continue

        quote_date = quote["date"].iloc[0]
        quote_age_days = int((as_of_date - quote_date).days)
        contract_month_date = quote["contractMonth"].iloc[0]

        if quote_age_days > max_quote_age_days or contract_month_date < front_month_floor:
            continue

        curve_rows.append(
            {
                "contractCode": contract_code,
                "contractMonth": contract_month_date,
                "contractLabel": quote["contractLabel"].iloc[0],
                "quoteDate": quote_date,
                "quoteAgeDays": quote_age_days,
                "close": quote["close"].iloc[0],
                "volume": quote["volume"].iloc[0],
                "openInterest": quote["open_interest"].iloc[0],
            }
        )

    curve = pd.DataFrame(curve_rows).sort_values("contractMonth").reset_index(drop=True)
    return curve.head(max_contracts)

term_structure_df = pd.DataFrame()
term_structure_as_of = None
for as_of_candidate in candidate_dates:
    candidate_curve = build_term_structure(pd.Timestamp(as_of_candidate))
    if len(candidate_curve) >= min_curve_points:
        term_structure_df = candidate_curve
        term_structure_as_of = pd.Timestamp(as_of_candidate)
        break

if term_structure_df.empty:
    term_structure_as_of = pd.Timestamp(candidate_dates[0])
    term_structure_df = build_term_structure(term_structure_as_of)

if term_structure_df.empty:
    raise RuntimeError(
        f"TurtleTrader data for {selected_turtle_market} did not produce a usable term structure curve. "
        f"Try increasing max_quote_age_days or selecting another market."
    )

term_structure_fig = go.Figure()
term_structure_fig.add_trace(
    go.Scatter(
        x=term_structure_df["contractLabel"],
        y=term_structure_df["close"],
        mode="lines+markers",
        name=selected_turtle_market,
        line={"color": "#22c55e", "width": 2.5},
        marker={"size": 8, "color": "#16a34a"},
        customdata=term_structure_df[["contractCode", "quoteDate", "quoteAgeDays", "volume", "openInterest"]],
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Close: %{y:.2f}<br>"
            "Contract: %{customdata[0]}<br>"
            "Quote Date: %{customdata[1]|%Y-%m-%d}<br>"
            "Quote Age: %{customdata[2]} days<br>"
            "Volume: %{customdata[3]:,.0f}<br>"
            "Open Interest: %{customdata[4]:,.0f}<extra></extra>"
        ),
    )
)

term_structure_fig.update_layout(
    title=(
        f"{selected_turtle_market} TurtleTrader futures term structure "
        f"(archive snapshot as of {term_structure_as_of:%Y-%m-%d})"
    ),
    xaxis_title="Contract Month",
    yaxis_title="Price",
    template="plotly_white",
    height=550,
    hovermode="x unified",
)

print(
    f"Using TurtleTrader archive data from {zip_url}. "
    f"Curve snapshot date: {term_structure_as_of:%Y-%m-%d}. "
    f"Contracts plotted: {len(term_structure_df)}."
)

term_structure_fig.show()

In [ ]:
#Load parameters
market_benchmark = yf.Ticker("SPY").history(period=period)
commodities_benchmark_broad = yf.Ticker("DBC").history(period=period)
commodities_benchmark_energy = yf.Ticker("DBE").history(period=period)
commodities_benchmark_industrial_metals = yf.Ticker("DBB").history(period=period)
commodities_benchmark_precious_metals = yf.Ticker("DBP").history(period=period)
commodities_benchmark_rare_earth_metals = yf.Ticker("EVMT").history(period=period)
commodities_benchmark_agriculture = yf.Ticker("DBA").history(period=period)

commodities_dashboard = {
    "Agriculture": {
        "Grains": {
            "Wheat": "ZW=F",
            "Corn": "ZC=F",
            "Soybeans": "ZS=F",
            "Soybean Oil": "ZL=F",
            "Soybean Meal": "ZM=F",
            "Oats": "ZO=F",
            "Rough Rice": "ZR=F"
        },
        "Softs": {
            "Coffee": "KC=F",
            "Cocoa": "CC=F",
            "Sugar": "SB=F",
            "Cotton": "CT=F",
            "Orange Juice": "OJ=F"
        },
        "Livestock": {
            "Live Cattle": "LE=F",
            "Feeder Cattle": "GF=F",
            "Lean Hogs": "HE=F"
        }
    },
    "Energy": {
        "Crude Oil": "CL=F",
        "Natural Gas": "NG=F",
        "Heating Oil": "HO=F",
        "Gasoline": "RB=F"
    },
    "Industrial / Construction Materials": {
        "Copper": "HG=F",
        "Aluminum": "ALI=F",   # optional ETF for tracking if needed
        "Zinc": "ZN=F",
        "Lumber": "LBR=F"       # housing/construction cycle commodity
    },
    "Tech / Battery / Rare Earth Metals": {
        "Lithium ETF": "LIT",
        "Rare Earth ETF": "REMX"
    },
    "Precious Metals": {
        "Gold": "GC=F",
        "Silver": "SI=F",
        "Platinum": "PL=F",
        "Palladium": "PA=F"
    }
}


# Extract dictionaries from commodities_dashboard
commodities_dict_energy = commodities_dashboard["Energy"]
commodities_dict_industrial_metals = commodities_dashboard["Industrial / Construction Materials"]
commodities_rare_earth_metals = commodities_dashboard["Tech / Battery / Rare Earth Metals"]
commodities_dict_precious_metals = commodities_dashboard["Precious Metals"]
commodities_dict_agriculture_grains = commodities_dashboard["Agriculture"]["Grains"]
commodities_livestock = commodities_dashboard["Agriculture"]["Livestock"]
commodities_dict_agriculture_softs = commodities_dashboard["Agriculture"]["Softs"]

#turn commodities_dict_energy into a dataframe with closing prices
commodities_df_energy = pd.DataFrame()

for key, value in commodities_dict_energy.items():
    commodities_df_energy[key] = yf.Ticker(value).history(period=period)["Close"]
    
#turn commodities_dict_industrial_metals into a dataframe with closing prices
commodities_df_metals = pd.DataFrame()

for key, value in commodities_dict_industrial_metals.items():
    commodities_df_metals[key] = yf.Ticker(value).history(period=period)["Close"]
    
for key, value in commodities_rare_earth_metals.items():
    commodities_df_metals[key] = yf.Ticker(value).history(period=period)["Close"]


#turn commodities_dict_precious_metals into a dataframe with closing prices
commodities_df_precious_metals = pd.DataFrame()

for key, value in commodities_dict_precious_metals.items():
    commodities_df_precious_metals[key] = yf.Ticker(value).history(period=period)["Close"]
    
#turn commodities_dict_agriculture_grains into a dataframe with closing prices
commodities_df_grains = pd.DataFrame()

for key, value in commodities_dict_agriculture_grains.items():
    commodities_df_grains[key] = yf.Ticker(value).history(period=period)["Close"]

#turn commodities_livestock into a dataframe with closing prices
commodities_df_livestock = pd.DataFrame()

for key, value in commodities_livestock.items():
    commodities_df_livestock[key] = yf.Ticker(value).history(period=period)["Close"]

#turn commodities_dict_agriculture_softs into a dataframe with closing prices
commodities_df_softs = pd.DataFrame()

for key, value in commodities_dict_agriculture_softs.items():
    commodities_df_softs[key] = yf.Ticker(value).history(period=period)["Close"]

time_frame_week = 7
time_frame_short = 21
time_frame_mid   = 50
time_frame_long = 200

#plot all commodities benchmarks



In [ ]:
#plot broad commodities benchmark
fig = make_subplots(rows=1, cols=1, subplot_titles=["Commodities Benchmarks"])
fig.add_trace(go.Scatter(x=commodities_benchmark_broad.index, y=commodities_benchmark_broad['Close'], mode='lines', name='Broad Commodities (DBC)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_benchmark_energy.index, y=commodities_benchmark_energy['Close'], mode='lines', name='Energy Commodities (DBE)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_benchmark_industrial_metals.index, y=commodities_benchmark_industrial_metals['Close'], mode='lines', name='Industrial Metals (DBB)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_benchmark_precious_metals.index, y=commodities_benchmark_precious_metals['Close'], mode='lines', name='Precious Metals (DBP)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_benchmark_rare_earth_metals.index, y=commodities_benchmark_rare_earth_metals['Close'], mode='lines', name='Rare Earth Metals (EVMT)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_benchmark_agriculture.index, y=commodities_benchmark_agriculture['Close'], mode='lines', name='Agriculture Commodities (DBA)'), row=1, col=1)
fig.update_layout(height=1200, title_text="Commodities Benchmarks Over Time")
fig.show()

In [ ]:
#plot energy commodities
fig = make_subplots(rows=1, cols=1, subplot_titles=["Energy Commodities"])
fig.add_trace(go.Scatter(x=commodities_df_energy.index, y=commodities_df_energy['Crude Oil'], mode='lines', name='Crude Oil (CL=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_energy.index, y=commodities_df_energy['Natural Gas'], mode='lines', name='Natural Gas (NG=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_energy.index, y=commodities_df_energy['Heating Oil'], mode='lines', name='Heating Oil (HO=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_energy.index, y=commodities_df_energy['Gasoline'], mode='lines', name='Gasoline (RB=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Energy Commodities Over Time")
fig.show()

In [ ]:
#plot industrial metals commodities
fig = make_subplots(rows=1, cols=1, subplot_titles=["Industrial / Construction Materials Commodities"])
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Copper'], mode='lines', name='Copper (HG=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Aluminum'], mode='lines', name='Aluminum (ALI=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Zinc'], mode='lines', name='Zinc (ZN=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Lumber'], mode='lines', name='Lumber (LBR=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Industrial / Construction Materials Commodities Over Time")
fig.show()


In [ ]:
#plot rare earth metals commodities
fig = make_subplots(rows=1, cols=1, subplot_titles=["Tech / Battery / Rare Earth Metals Commodities"])
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Lithium ETF'], mode='lines', name='Lithium ETF (LIT)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Rare Earth ETF'], mode='lines', name='Rare Earth ETF (REMX)'), row=1, col=1)
fig.update_layout(height=600, title_text="Tech / Battery / Rare Earth Metals Commodities Over Time")
fig.show()


In [ ]:
#plot precious metals commodities
fig = make_subplots(rows=1, cols=1, subplot_titles=["Precious Metals Commodities"])
fig.add_trace(go.Scatter(x=commodities_df_precious_metals.index, y=commodities_df_precious_metals['Gold'], mode='lines', name='Gold (GC=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_precious_metals.index, y=commodities_df_precious_metals['Silver'], mode='lines', name='Silver (SI=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_precious_metals.index, y=commodities_df_precious_metals['Platinum'], mode='lines', name='Platinum (PL=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_precious_metals.index, y=commodities_df_precious_metals['Palladium'], mode='lines', name='Palladium (PA=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Precious Metals Commodities Over Time")
fig.show()


In [ ]:
#plot agriculture commodities - grains    
fig = make_subplots(rows=1, cols=1, subplot_titles=["Agriculture Commodities - Grains"])
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Wheat'], mode='lines', name='Wheat (ZW=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Corn'], mode='lines', name='Corn (ZC=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Soybeans'], mode='lines', name='Soybeans (ZS=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Soybean Oil'], mode='lines', name='Soybean Oil (ZL=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Soybean Meal'], mode='lines', name='Soybean Meal (ZM=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Oats'], mode='lines', name='Oats (ZO=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Rough Rice'], mode='lines', name='Rough Rice (ZR=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Agriculture Commodities - Grains Over Time")

fig.show()

In [ ]:
#plot agriculture commodities - softs
fig = make_subplots(rows=1, cols=1, subplot_titles=["Agriculture Commodities - Softs"])
fig.add_trace(go.Scatter(x=commodities_df_softs.index, y=commodities_df_softs['Coffee'], mode='lines', name='Coffee (KC=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_softs.index, y=commodities_df_softs['Cocoa'], mode='lines', name='Cocoa (CC=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_softs.index, y=commodities_df_softs['Sugar'], mode='lines', name='Sugar (SB=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_softs.index, y=commodities_df_softs['Cotton'], mode='lines', name='Cotton (CT=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_softs.index, y=commodities_df_softs['Orange Juice'], mode='lines', name='Orange Juice (OJ=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Agriculture Commodities - Softs Over Time")
fig.show()


In [ ]:
#plot agriculture commodities - livestock
fig = make_subplots(rows=1, cols=1, subplot_titles=["Agriculture Commodities - Livestock"])
fig.add_trace(go.Scatter(x=commodities_df_livestock.index, y=commodities_df_livestock['Live Cattle'], mode='lines', name='Live Cattle (LE=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_livestock.index, y=commodities_df_livestock['Feeder Cattle'], mode='lines', name='Feeder Cattle (GF=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_livestock.index, y=commodities_df_livestock['Lean Hogs'], mode='lines', name='Lean Hogs (HE=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Agriculture Commodities - Livestock Over Time")
fig.show()
